Attention- Multi Head Attention

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tiktoken

ids = torch.load('../../datasets/ids.pt')

print(ids[:50])

[17816, 3041, 344, 21749, 25, 3256, 705, 26979, 1136, 540, 3256, 705, 6732, 715, 1045, 3256, 705, 4480, 3256, 705, 7376, 2771, 3256, 705, 41222, 25, 23, 3256, 705, 11664, 3256, 705, 33856, 82, 3256, 705, 82, 30945, 3256, 705, 12945, 268, 3256, 46083, 3256, 705, 16, 3256, 705, 83]


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`, 
        # this will result in errors in the mask creation further below. 
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

embed_dim = 256
fenster = 3

token_embedding = nn.Embedding(50257, embed_dim)
pos_embedding = nn.Embedding(fenster, embed_dim)

# einen batch manuell erstellen
token_ids = torch.tensor(ids[:6]).unsqueeze(0)  # shape: [1, 6]
token_ids = token_ids[:, :fenster]              # shape: [1, 3]

token_vektoren = token_embedding(token_ids)
pos_vektoren = pos_embedding(torch.arange(fenster))
x = token_vektoren + pos_vektoren
print("x shape:", x.shape)


d_in = 256        
d_out = 256
context_length = 3  
num_heads = 4

mha = MultiHeadAttention(d_in, d_out, context_length, dropout=0.0, num_heads=num_heads)

context_vecs = mha(x)  
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)




x shape: torch.Size([1, 3, 256])
tensor([[[-7.6414e-01,  2.9154e-01, -1.9037e-01,  3.1077e-01,  3.5080e-01,
           9.9317e-02,  5.0221e-02,  3.0308e-01,  1.9244e-01,  4.7636e-02,
          -7.1880e-02, -2.9143e-01,  5.2602e-01,  7.5202e-02,  2.2773e-01,
          -2.0110e-01, -6.3697e-02, -2.8900e-01, -3.3126e-01, -4.2194e-01,
          -2.3386e-01,  4.5508e-02,  5.2981e-01,  4.4535e-01,  4.4046e-01,
           4.3316e-01, -5.2813e-01, -4.7818e-01, -1.5627e-01,  6.0605e-01,
          -8.3516e-02,  2.9352e-01,  7.0454e-01,  6.9913e-02, -3.4901e-01,
          -2.1434e-01, -6.7013e-01,  3.9920e-01, -7.8823e-01,  6.7225e-02,
           3.5327e-01, -2.5369e-01,  3.9062e-01,  3.3338e-01,  2.6941e-01,
          -1.4387e-01,  8.0987e-01,  9.1302e-02, -2.2594e-01, -3.0039e-01,
           6.6410e-01, -1.1495e-01,  1.0442e-01,  5.6625e-01, -4.3983e-01,
          -5.6057e-01,  8.9655e-01,  8.2673e-04, -5.6083e-02, -3.8740e-01,
           5.6725e-01, -4.5790e-01, -5.8352e-01, -4.8122e-01, -4.13

In [5]:
torch.save(context_vecs, '../../datasets/context_vecs.pt')